# The Nature of Randomness: From Pseudorandom to Quantum

**Why not all "random" is created equal — and why it matters for QuantumFair**

---

This notebook explores the fundamental differences between types of randomness used in computing, builds up to quantum random number generation (QRNG), and explains why von Neumann debiasing is essential when working with real quantum hardware.

## 1. What Is Randomness?

A sequence is **random** if no algorithm can predict the next value given all previous values. This sounds simple, but the definition splits into two fundamentally different categories:

| Property | Pseudorandom | True Random |
|---|---|---|
| **Source** | Deterministic algorithm | Physical process |
| **Reproducible?** | Yes (same seed → same sequence) | No (never reproducible) |
| **Predictable?** | Yes (if you know the state) | No (not even in principle) |
| **Period** | Finite (eventually repeats) | Infinite (no period) |
| **Speed** | Very fast | Limited by physics |

The key insight: **pseudorandomness is a simulation of randomness, not the real thing.** It's a deterministic sequence that *looks* random to statistical tests but is entirely predetermined the moment the seed is chosen.

## 2. Level 0 — Software PRNGs (`Math.random()`)

The most common source of "randomness" in software. Salesforce Apex's `Math.random()`, JavaScript's `Math.random()`, Python's `random` module — all use **pseudorandom number generators (PRNGs)**.

### How They Work

A PRNG is a pure function: `state → (output, next_state)`. The most widely used algorithm is the **Mersenne Twister (MT19937)**, which Python's `random` module uses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Python's random module uses Mersenne Twister
import random

# Set a seed - this makes the sequence COMPLETELY DETERMINISTIC
random.seed(42)

sequence_a = [random.randint(0, 1) for _ in range(1000)]

# Reset the same seed - we get the EXACT SAME sequence
random.seed(42)

sequence_b = [random.randint(0, 1) for _ in range(1000)]

print(f"Sequences identical: {sequence_a == sequence_b}")
print(f"First 40 bits: {''.join(map(str, sequence_a[:40]))}")
print(f"\nThis sequence LOOKS random but is 100% predetermined.")

### The Problem with PRNGs for Fairness

PRNGs have a critical property: **anyone who knows the internal state can predict every future output.** The Mersenne Twister has a state of 624 32-bit integers (19,968 bits). If an observer collects 624 consecutive outputs, they can reconstruct the full state and predict all future values.

For lead assignment in Salesforce:
- A sales rep who reverse-engineers the PRNG state could predict which leads they'll receive
- An auditor cannot prove the assignments weren't manipulated (the admin could have chosen a seed that produces a desired outcome)
- There is no physical evidence that the process was fair

**PRNGs are tools for simulation, not for fairness.**

In [ ]:
# Demonstration: Mersenne Twister state recovery
# With just 624 outputs, the ENTIRE future sequence is predictable

random.seed()  # "unknown" seed

# Attacker observes 624 consecutive 32-bit outputs
observed = [random.getrandbits(32) for _ in range(624)]

# In practice, tools like 'randcrack' recover the full MT state from these 624 values
# and predict all future outputs with 100% accuracy

print(f"Mersenne Twister state size: 624 × 32 = {624*32:,} bits")
print(f"Outputs needed to crack:     624 values")
print(f"After cracking:              Every future output is known")
print(f"\nThis is why PRNG-based 'fairness' is an illusion.")

## 3. Level 1 — CSPRNGs (Cryptographically Secure PRNGs)

Cryptographic applications need PRNGs that resist state recovery even if an attacker sees many outputs. **CSPRNGs** like ChaCha20, AES-CTR, and `/dev/urandom` are designed so that observing outputs doesn't reveal the internal state.

However, CSPRNGs are still **deterministic algorithms**. They need to be *seeded* with true entropy, and their security relies on computational hardness assumptions (e.g., "AES is hard to invert"). They are not truly random — they are *computationally indistinguishable from random* under certain assumptions.

| CSPRNG | Used By | Seed Source |
|---|---|---|
| ChaCha20 | Linux `/dev/urandom`, OpenSSL | Hardware entropy pool |
| AES-CTR-DRBG | NIST SP 800-90A, many TLS stacks | OS entropy |
| Fortuna | macOS, FreeBSD | Multiple entropy pools |

In [ ]:
import secrets
import os

# Python's 'secrets' module uses the OS CSPRNG (/dev/urandom or CryptGenRandom)
csprng_bits = ''.join(str(secrets.randbelow(2)) for _ in range(40))
print(f"CSPRNG output:  {csprng_bits}")

# os.urandom gives raw bytes from the OS entropy pool
raw_entropy = os.urandom(5)
print(f"OS entropy:     {''.join(f'{b:08b}' for b in raw_entropy)}")

print(f"\nCSPRNG: unpredictable in PRACTICE (computationally secure)")
print(f"But still deterministic — security depends on AES/ChaCha being 'hard'")
print(f"A sufficiently powerful computer (or quantum computer) could break it")

## 4. Level 2 — Hardware RNG: Intel RDRAND

Modern CPUs include hardware random number generators. Intel's **RDRAND** instruction (introduced with Ivy Bridge, 2012) samples a hardware entropy source on the CPU die.

### How RDRAND Works

```
┌─────────────────────────────────────────────────┐
│                    CPU Die                       │
│                                                  │
│  ┌──────────────┐    ┌──────────────────────┐   │
│  │ Thermal Noise │───▶│ AES-CBC-MAC CSPRNG   │   │
│  │ (Analog)      │    │ (Conditioner)        │   │
│  └──────────────┘    └──────────┬───────────┘   │
│                                  │               │
│                                  ▼               │
│                          RDRAND instruction      │
│                          (64 bits per call)      │
└─────────────────────────────────────────────────┘
```

The entropy source is **thermal noise** in transistor junctions — a physical (classical) process. This is then conditioned through AES-CBC-MAC to produce uniform output.

### The Trust Problem

RDRAND is a **black box on a proprietary chip**:

- The analog entropy source cannot be independently audited
- The conditioning algorithm is AES-based — if the AES key is known to Intel (or an intelligence agency), all outputs are predictable
- In 2019, a [vulnerability](https://en.wikipedia.org/wiki/RDRAND#Reception) was discovered where RDRAND could return predictable values under certain microarchitectural conditions
- Linux kernel developers initially refused to trust RDRAND as a sole entropy source (Linus Torvalds accepted it only as one input to the entropy pool, never the only one)

RDRAND produces **probably-random** numbers from a **probably-working** entropy source through a **probably-unbackdoored** conditioner. Each "probably" is a trust assumption, not a physical guarantee.

In [ ]:
# We can't call RDRAND directly from Python, but we can illustrate
# the fundamental difference between hardware RNG and true randomness

print("RDRAND Trust Chain:")
print("="*60)
print("1. Thermal noise on CPU die        → Trust Intel's analog circuit")
print("2. AES-CBC-MAC conditioning         → Trust the AES key is secret")
print("3. RDRAND instruction output        → Trust no microarch bugs")
print("4. OS entropy pool mixing           → Trust the OS implementation")
print("5. Application receives 'random'    → Trust the full chain above")
print()
print("Each layer is a TRUST ASSUMPTION, not a provable guarantee.")
print("An auditor cannot verify any of these layers independently.")
print()
print("Compare to quantum RNG:")
print("="*60)
print("1. Qubit in superposition           → Guaranteed by quantum mechanics")
print("2. Measurement collapses to 0 or 1  → Fundamentally unpredictable")
print("3. Bell test violation (S > 2)       → PROVES quantum origin")
print("4. Raw bitstring recorded            → Auditable via IBM job ID")
print()
print("Quantum randomness is provable. Hardware RNG randomness is trusted.")

## 5. Level 3 — Simulated Quantum Randomness

Quantum computing simulators (like Qiskit's `AerSimulator` or `StatevectorSimulator`) can simulate quantum circuits on classical hardware. When you "measure" a simulated qubit, the simulator uses a classical PRNG to sample from the probability distribution.

**This is not quantum randomness.** It's a classical PRNG sampling from a mathematically-computed probability distribution. The output is deterministic given the simulator's PRNG seed.

This is a crucial distinction that many people miss: **simulating a quantum computer does not produce quantum randomness, just like simulating a coin flip doesn't produce a physical coin flip.**

In [ ]:
# Simulating a quantum Hadamard + measurement
# This is NOT quantum randomness - it's classical sampling

def simulated_hadamard_measurement(num_bits, num_shots, seed=None):
    """Simulate a quantum H|0⟩ → measure circuit.
    
    A real Hadamard gate puts a qubit into equal superposition:
    H|0⟩ = (|0⟩ + |1⟩) / √2
    
    Measurement then collapses to |0⟩ or |1⟩ with 50% probability each.
    
    But HERE, we're just calling np.random — a classical PRNG.
    The quantum mechanics is simulated, not real.
    """
    rng = np.random.RandomState(seed)
    # This is just classical coin flipping dressed up as quantum
    return rng.randint(0, 2, size=(num_shots, num_bits))

# Same seed = same "quantum" results. Not random at all.
result_a = simulated_hadamard_measurement(8, 5, seed=42)
result_b = simulated_hadamard_measurement(8, 5, seed=42)

print("Simulated 'quantum' measurement (seed=42):")
for i, row in enumerate(result_a):
    print(f"  Shot {i}: {''.join(map(str, row))}")

print(f"\nIdentical with same seed: {np.array_equal(result_a, result_b)}")
print(f"\nThis is NOT quantum randomness. It's np.random with extra steps.")

In [ ]:
# The mathematical model is correct - probabilities are right.
# But the SOURCE of randomness is a PRNG, not physics.

results = simulated_hadamard_measurement(1, 100_000, seed=None)
ones = results.sum()
zeros = len(results) - ones

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(['0', '1'], [zeros, ones], color=['#2196F3', '#FF5722'], edgecolor='black')
ax1.set_title('Simulated Quantum Measurement\n(Classical PRNG sampling from |ψ⟩ = (|0⟩+|1⟩)/√2)', fontsize=11)
ax1.set_ylabel('Count')
ax1.axhline(y=50_000, color='gray', linestyle='--', alpha=0.7, label='Expected (50%)')
ax1.legend()

# Show that simulated results pass statistical tests
# BUT so does any decent PRNG — passing tests doesn't prove true randomness
proportions = []
for trial in range(200):
    trial_results = simulated_hadamard_measurement(1, 1000)
    proportions.append(trial_results.mean())

ax2.hist(proportions, bins=30, color='#9C27B0', edgecolor='black', alpha=0.8)
ax2.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Expected mean')
ax2.set_title('Distribution of Measured Proportions\n(200 trials × 1000 shots)', fontsize=11)
ax2.set_xlabel('Proportion of |1⟩')
ax2.set_ylabel('Frequency')
ax2.legend()

plt.tight_layout()
plt.savefig('simulated_vs_real.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Proportion of 1s: {ones/len(results):.4f} (expected: 0.5000)")
print(f"\nThe statistics look perfect. But statistics can't distinguish")
print(f"a good PRNG from true randomness. That's the whole point.")

## 6. Level 4 — Real Quantum RNG (What QuantumFair Does)

When a qubit on a real quantum processor (like IBM's Heron r2) is placed in superposition and measured, the outcome is **fundamentally undetermined** until the moment of measurement. This is not a gap in our knowledge — it is a property of nature established by:

1. **The Born Rule** — Measurement outcomes are probabilistic, with probabilities given by |⟨ψ|φ⟩|²
2. **Bell's Theorem (1964)** — No local hidden variable theory can reproduce all predictions of quantum mechanics
3. **Experimental Bell test violations** — Confirmed in loophole-free experiments (Hensen et al. 2015, Nobel Prize 2022)

### The Circuit

For a single random bit:

```
        ┌───┐ ┌─┐
|0⟩ ────┤ H ├─┤M├──── → 0 or 1 (truly random)
        └───┘ └─┘
```

The Hadamard gate transforms the initial state:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

Measurement then collapses the superposition:
- P(0) = |⟨0|ψ⟩|² = 1/2
- P(1) = |⟨1|ψ⟩|² = 1/2

**No information in the universe — not the state of the qubit, not the state of the lab, not the laws of physics themselves — determines which outcome will occur.** This is the strongest possible form of randomness.

In [ ]:
# Visualize the quantum state on the Bloch sphere

fig = plt.figure(figsize=(14, 5))

# State |0⟩ (before Hadamard)
ax1 = fig.add_subplot(131)
circle1 = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax1.add_patch(circle1)
ax1.annotate('', xy=(0, 0.9), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#2196F3', lw=3))
ax1.text(0, 1.15, '|0⟩', ha='center', fontsize=14, fontweight='bold')
ax1.text(0, -1.15, '|1⟩', ha='center', fontsize=14, color='gray')
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')
ax1.set_title('Initial state: |0⟩\n(deterministic)', fontsize=12)
ax1.axis('off')

# State after Hadamard: (|0⟩+|1⟩)/√2
ax2 = fig.add_subplot(132)
circle2 = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax2.add_patch(circle2)
ax2.annotate('', xy=(0.9, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#FF5722', lw=3))
ax2.text(1.15, 0, '|+⟩', ha='left', fontsize=14, fontweight='bold', color='#FF5722')
ax2.text(0, 1.15, '|0⟩', ha='center', fontsize=14, color='gray')
ax2.text(0, -1.15, '|1⟩', ha='center', fontsize=14, color='gray')
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.set_title('After H gate: (|0⟩+|1⟩)/√2\n(superposition — on equator)', fontsize=12)
ax2.axis('off')

# Measurement: collapse
ax3 = fig.add_subplot(133)
circle3 = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=2)
ax3.add_patch(circle3)
# Show both possibilities with dashed arrows
ax3.annotate('', xy=(0, 0.9), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#2196F3', lw=2, linestyle='dashed'))
ax3.annotate('', xy=(0, -0.9), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#FF5722', lw=2, linestyle='dashed'))
ax3.text(0.15, 0.7, '50%', fontsize=12, color='#2196F3', fontweight='bold')
ax3.text(0.15, -0.7, '50%', fontsize=12, color='#FF5722', fontweight='bold')
ax3.text(0, 1.15, '|0⟩ → bit 0', ha='center', fontsize=13, fontweight='bold', color='#2196F3')
ax3.text(0, -1.15, '|1⟩ → bit 1', ha='center', fontsize=13, fontweight='bold', color='#FF5722')
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)
ax3.set_aspect('equal')
ax3.set_title('Measurement: collapse\n(fundamentally unpredictable)', fontsize=12)
ax3.axis('off')

plt.suptitle('Quantum Random Bit Generation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('bloch_sphere_rng.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Why Von Neumann Debiasing?

### The Problem: Hardware Bias

The theory says P(0) = P(1) = 0.5 exactly. Real quantum hardware disagrees.

On a physical qubit, several error sources introduce **bias** — a systematic tendency to measure one outcome more than the other:

| Error Source | Effect | Typical Magnitude |
|---|---|---|
| **Readout error** | Detector miscounts 0↔1 | 0.5%–3% per qubit |
| **Gate error** | Hadamard rotation is imperfect | 0.01%–0.1% |
| **T1 decay** | Qubit decays toward \|0⟩ during measurement | Varies by qubit |
| **Crosstalk** | Neighboring qubits influence each other | 0.1%–1% |

On IBM's Heron r2 (`ibm_kingston`), a single qubit might produce 51.2% zeros and 48.8% ones instead of a perfect 50/50. This bias is **small but systematic** — and for fairness applications, any bias is unacceptable.

### The Solution: Von Neumann's 1951 Method

John von Neumann published an elegant solution in 1951. The insight: even if a coin is biased, the probability of getting **01** is the same as getting **10**.

If P(0) = p and P(1) = 1-p, then:
- P(01) = p(1-p)
- P(10) = (1-p)p
- These are **identical** regardless of p!

**Algorithm:**
1. Take bits in pairs
2. If pair is **01** → output **0**
3. If pair is **10** → output **1**
4. If pair is **00** or **11** → discard (carry no information)

The output is **perfectly unbiased** regardless of the input bias.

In [ ]:
def von_neumann_debias(bits):
    """Apply von Neumann debiasing to a sequence of bits.
    
    Takes pairs of bits:
        01 → 0
        10 → 1
        00 → discard
        11 → discard
    
    Output is perfectly unbiased regardless of input bias.
    Trade-off: ~75% of bits are discarded (for 50% bias).
    """
    output = []
    discarded = 0
    for i in range(0, len(bits) - 1, 2):
        a, b = bits[i], bits[i+1]
        if a == 0 and b == 1:
            output.append(0)
        elif a == 1 and b == 0:
            output.append(1)
        else:
            discarded += 1
    return output, discarded


# Simulate a biased quantum source (e.g., readout error)
# Real qubit might have P(0) = 0.52 instead of 0.50
bias = 0.52
n_bits = 100_000

biased_bits = (np.random.random(n_bits) > bias).astype(int).tolist()

debiased_bits, num_discarded = von_neumann_debias(biased_bits)

biased_ones = sum(biased_bits) / len(biased_bits)
debiased_ones = sum(debiased_bits) / len(debiased_bits) if debiased_bits else 0

print(f"Input bias:     P(1) = {biased_ones:.4f}  (expected: {1-bias:.4f}, error: {abs(biased_ones - 0.5):.4f})")
print(f"After debiasing: P(1) = {debiased_ones:.4f}  (expected: 0.5000, error: {abs(debiased_ones - 0.5):.4f})")
print(f"")
print(f"Input bits:      {n_bits:,}")
print(f"Output bits:     {len(debiased_bits):,}")
print(f"Discarded pairs: {num_discarded:,}")
print(f"Efficiency:      {len(debiased_bits) / (n_bits/2) * 100:.1f}%")

In [ ]:
# Demonstrate that debiasing works across different bias levels

biases = np.arange(0.30, 0.71, 0.02)
raw_proportions = []
debiased_proportions = []
efficiencies = []

for p_zero in biases:
    bits = (np.random.random(50_000) > p_zero).astype(int).tolist()
    raw_proportions.append(sum(bits) / len(bits))
    
    db, disc = von_neumann_debias(bits)
    if db:
        debiased_proportions.append(sum(db) / len(db))
        efficiencies.append(len(db) / (len(bits) / 2) * 100)
    else:
        debiased_proportions.append(0.5)
        efficiencies.append(0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(biases, raw_proportions, 'o-', color='#FF5722', label='Raw (biased)', markersize=4)
ax1.plot(biases, debiased_proportions, 's-', color='#4CAF50', label='After von Neumann', markersize=4)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Ideal (0.5)')
ax1.fill_between(biases, 0.49, 0.51, alpha=0.1, color='green', label='±1% tolerance')
ax1.set_xlabel('P(0) of biased source', fontsize=12)
ax1.set_ylabel('Measured P(1)', fontsize=12)
ax1.set_title('Von Neumann Debiasing Corrects Any Bias', fontsize=13)
ax1.legend(fontsize=10)
ax1.set_ylim(0.25, 0.75)
ax1.grid(True, alpha=0.3)

ax2.plot(biases, efficiencies, 'D-', color='#9C27B0', markersize=4)
ax2.set_xlabel('P(0) of biased source', fontsize=12)
ax2.set_ylabel('Efficiency (%)', fontsize=12)
ax2.set_title('Debiasing Efficiency\n(% of input pairs that produce output)', fontsize=13)
ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Unbiased source')
ax2.annotate(f'Peak at p=0.5\n~50% efficiency', xy=(0.5, max(efficiencies)), 
            xytext=(0.58, max(efficiencies)-5),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('von_neumann_debiasing.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key insight: the green line is FLAT at 0.5 regardless of input bias.")
print("The cost is efficiency — more biased sources waste more bits.")
print(f"For IBM hardware (bias ~0.52): efficiency ≈ {efficiencies[len(biases)//2]:.0f}%")
print("We compensate by requesting more shots from the quantum computer.")

### Why Not Just Calibrate Away the Bias?

You might ask: if we know a qubit is biased toward 0, why not just adjust the probabilities mathematically?

Three reasons:

1. **Bias drifts.** Qubit error rates change with temperature, calibration cycles, and neighboring circuit activity. The bias you measured 10 minutes ago may not be the bias right now.

2. **Calibration requires trust.** If we calibrate, we're introducing a correction factor that someone chose. An auditor would need to verify the calibration was honest — defeating the purpose of provable fairness.

3. **Von Neumann debiasing is unconditional.** It works for ANY bias value without knowing what the bias is. It requires zero trust in calibration data. The math is provably correct regardless of hardware behavior.

This is why QuantumFair uses von Neumann debiasing: **it converts imperfect hardware into a perfect random source with a mathematical guarantee, not a calibration assumption.**

## 8. The Bell Test: Proving Quantum Origin

Statistical tests (frequency, runs, chi-squared) can verify that a sequence *looks* random — but they cannot distinguish a good PRNG from true quantum randomness. Both pass the same tests.

The **CHSH Bell test** can. It's a physics experiment that produces a measurable quantity (S) with a hard ceiling for any classical system:

$$S_{\text{classical}} \leq 2 \quad \text{(Bell's inequality)}$$
$$S_{\text{quantum}} = 2\sqrt{2} \approx 2.828 \quad \text{(Tsirelson's bound)}$$

If S > 2, the randomness **cannot have come from any classical source**. Period. This is not a statistical argument — it's a theorem of physics.

### How It Works

1. Create an entangled Bell pair: $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$
2. Each qubit is measured in one of two randomly chosen bases
3. Correlations between measurement outcomes are computed for four basis combinations
4. The CHSH S parameter is calculated from these correlations
5. S > 2 proves non-classical correlations → proves quantum origin of randomness

In [ ]:
def simulate_chsh_experiment(n_shots=10000, quantum=True):
    """
    Simulate a CHSH Bell test experiment.
    
    If quantum=True: uses quantum mechanical predictions (violates Bell inequality)
    If quantum=False: uses classical correlations (respects Bell inequality)
    """
    # Alice's measurement angles
    a1, a2 = 0, np.pi/4              # 0° and 45°
    # Bob's measurement angles  
    b1, b2 = np.pi/8, 3*np.pi/8      # 22.5° and 67.5°
    
    settings = [(a1, b1), (a1, b2), (a2, b1), (a2, b2)]
    correlations = []
    
    for a, b in settings:
        if quantum:
            # Quantum prediction: E(a,b) = -cos(2(a-b)) for |Φ+⟩
            E = -np.cos(2*(a - b))
        else:
            # Best classical strategy: E = cos(a-b) capped by Bell bound
            E = np.cos(a - b) * 0.7  # scaled to respect S ≤ 2
        
        # Add realistic noise
        E_noisy = E + np.random.normal(0, 0.02)
        correlations.append(E_noisy)
    
    # CHSH: S = |E(a1,b1) - E(a1,b2) + E(a2,b1) + E(a2,b2)|
    E11, E12, E21, E22 = correlations
    S = abs(E11 - E12 + E21 + E22)
    
    return S, correlations


# Run many trials
n_trials = 1000
quantum_S = [simulate_chsh_experiment(quantum=True)[0] for _ in range(n_trials)]
classical_S = [simulate_chsh_experiment(quantum=False)[0] for _ in range(n_trials)]

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(classical_S, bins=40, alpha=0.7, color='#2196F3', label='Classical source', edgecolor='black')
ax.hist(quantum_S, bins=40, alpha=0.7, color='#FF5722', label='Quantum source', edgecolor='black')
ax.axvline(x=2.0, color='red', linewidth=3, linestyle='--', label="Bell's limit (S = 2)")
ax.axvline(x=2*np.sqrt(2), color='purple', linewidth=2, linestyle=':', 
           label=f"Tsirelson's bound (S = 2√2 ≈ {2*np.sqrt(2):.3f})")

ax.fill_betweenx([0, ax.get_ylim()[1] or 100], 2, 3, alpha=0.08, color='red', label='Quantum-only region')

ax.set_xlabel('CHSH S parameter', fontsize=13)
ax.set_ylabel('Frequency', fontsize=13)
ax.set_title('Bell Test: Classical vs Quantum Sources\n(S > 2 is impossible without quantum mechanics)', fontsize=14)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bell_test_chsh.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Classical S: {np.mean(classical_S):.3f} ± {np.std(classical_S):.3f} (always ≤ 2)")
print(f"Quantum S:   {np.mean(quantum_S):.3f} ± {np.std(quantum_S):.3f} (violates Bell inequality)")
print(f"\nWhen QuantumFair reports S > 2 on a fairness certificate,")
print(f"it is PHYSICALLY IMPOSSIBLE that the randomness came from a classical source.")

## 9. The Complete Hierarchy of Randomness

Putting it all together:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

levels = [
    (1.0, '#BBDEFB', 'Level 0: Software PRNG',
     'Math.random(), Mersenne Twister, Apex Math.random()',
     'Deterministic. Predictable with seed knowledge.\nPeriod: 2¹⁹⁹³⁷-1 (finite). No physical entropy.'),
    (3.0, '#90CAF9', 'Level 1: CSPRNG',
     'ChaCha20, AES-CTR, /dev/urandom, secrets module',
     'Computationally unpredictable. Still deterministic.\nSecurity relies on hardness assumptions (AES, ChaCha).'),
    (5.0, '#64B5F6', 'Level 2: Hardware RNG (RDRAND)',
     'Intel RDRAND/RDSEED, TPM, thermal noise',
     'Physical entropy source. But: opaque, unauditable.\nClassical physics — in principle deterministic.'),
    (7.0, '#42A5F5', 'Level 3: Simulated Quantum',
     'Qiskit Aer, Cirq Simulator, Pennylane default.qubit',
     'Correct probability model, but PRNG sampling.\nNOT quantum randomness. Just math with extra steps.'),
    (9.0, '#FF7043', 'Level 4: Real Quantum RNG ← QuantumFair',
     'IBM Quantum (Heron r2), Quantinuum, IonQ',
     'Fundamentally unpredictable (Born rule).\nProvably non-classical (Bell test). Auditable (IBM job ID).')
]

for y, color, title, examples, desc in levels:
    rect = plt.Rectangle((0.3, y - 0.45), 9.4, 1.5, 
                         facecolor=color, edgecolor='black', 
                         linewidth=1.5, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(0.6, y + 0.65, title, fontsize=13, fontweight='bold', zorder=3, va='center')
    ax.text(0.6, y + 0.2, examples, fontsize=9, style='italic', color='#333', zorder=3, va='center')
    ax.text(0.6, y - 0.25, desc, fontsize=9, color='#222', zorder=3, va='center', linespacing=1.4)

# Arrow on the left side
ax.annotate('', xy=(0.15, 9.2), xytext=(0.15, 1.0),
           arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
ax.text(0.15, 5.0, 'Increasing\nrandomness\nquality', rotation=90, 
       ha='center', va='center', fontsize=10, fontweight='bold', color='#555')

plt.title('The Hierarchy of Randomness in Computing', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('randomness_hierarchy.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Conclusion: The Value of QuantumFair in Salesforce

### The Gap

Every Salesforce org that assigns leads, selects audit samples, runs incentive lotteries, or distributes cases uses `Math.random()` — a Level 0 PRNG. This means:

- **No proof of fairness.** An admin could restart the org with a seed that produces a desired outcome. No one would know.
- **No audit trail.** When a sales rep claims lead distribution is unfair, there is no evidence to confirm or deny it.
- **No compliance defense.** When regulators ask how audit samples were selected, "we used Math.random()" is not a defensible answer.

### What QuantumFair Changes

QuantumFair replaces the weakest link (Level 0 PRNG) with the strongest possible source (Level 4 quantum RNG) and wraps it in an auditable certificate:

| Without QuantumFair | With QuantumFair |
|---|---|
| `Math.random()` → assignment | Quantum measurement → von Neumann debiasing → assignment |
| No record of how randomness was generated | Fairness Certificate with IBM job ID, circuit, and statistical validation |
| Fairness is claimed | Fairness is **evidenced**: replayable assignment, hash-chained certificate, optional Bell test (S > 2 by ≥2σ) |
| Admin could manipulate outcomes | Outcomes determined by quantum physics, not by anyone |
| Auditor must trust the software | Auditor can replay the assignment and look up the job via IBM's API |

### Why Open Source?

QuantumFair is an Open Systems Foundation Labs tech demo, Apache-2.0 licensed, because:

1. **Quantum computing needs real-world use cases.** The gap between quantum research and business applications is large. QuantumFair shows one concrete pattern inside a CRM platform.

2. **Trust requires transparency.** A fairness tool that hides its source code undermines its own value proposition. Open source means anyone can verify the circuit construction, debiasing, and certificate generation.

3. **No middleware.** BYOQ (Bring Your Own Quantum) means each org uses its own IBM Quantum account. No servers, no endpoints: Apex calls IBM's REST API directly. The app is pure Salesforce metadata.

### Honest limits

- QPU time is not free beyond IBM's Open Plan allowance; a classical CSPRNG is far cheaper.
- The Bell test is evidence *consistent with quantum origin*, not device-independent certification — the org still trusts IBM to run the circuit as reported.
- What makes an assignment auditable is the replayable, hash-chained record; the quantum source is what makes the randomness itself hard to predict or steer.


In [ ]:
# Final visualization: what a QuantumFair certificate represents

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Certificate border
outer = plt.Rectangle((0.5, 0.5), 9, 9, facecolor='#FFFDE7', 
                       edgecolor='#333', linewidth=3, zorder=1)
inner = plt.Rectangle((0.7, 0.7), 8.6, 8.6, facecolor='none', 
                       edgecolor='#999', linewidth=1, linestyle='--', zorder=2)
ax.add_patch(outer)
ax.add_patch(inner)

ax.text(5, 9.0, 'QUANTUMFAIR', ha='center', fontsize=20, fontweight='bold', 
       color='#1A237E', zorder=3, family='monospace')
ax.text(5, 8.4, 'Fairness Certificate', ha='center', fontsize=14, 
       color='#333', zorder=3)

# Horizontal line
ax.plot([1.5, 8.5], [8.0, 8.0], color='#1A237E', linewidth=1.5, zorder=3)

details = [
    ('Certificate ID:', 'QF-2026-00042'),
    ('Purpose:', 'Q1 Lead Assignment — West Region'),
    ('IBM Job ID:', 'c5dge2d3rn7breq27i9g'),
    ('Backend:', 'ibm_kingston (Heron r2, 156 qubits)'),
    ('Circuit:', '8-qubit H⊗8 |0⟩⊗8 → Measure (ISA-compiled)'),
    ('Shots:', '4,096'),
    ('Debiasing:', 'Von Neumann (efficiency: 49.8%)'),
    ('Random bits:', '16,220 raw → 8,082 debiased'),
    ('Frequency test:', '✓ PASS  (p = 0.847)'),
    ('Runs test:', '✓ PASS  (p = 0.623)'),
    ('Chi-squared:', '✓ PASS  (p = 0.391)'),
    ('CHSH S-value:', '2.714 > 2.000  ✓ QUANTUM CERTIFIED'),
]

y_pos = 7.5
for label, value in details:
    ax.text(1.2, y_pos, label, fontsize=9, fontweight='bold', color='#444', 
           zorder=3, family='monospace')
    color = '#2E7D32' if '✓' in value else '#333'
    if 'QUANTUM CERTIFIED' in value:
        color = '#1A237E'
    ax.text(4.5, y_pos, value, fontsize=9, color=color, 
           zorder=3, family='monospace', fontweight='bold' if '✓' in value else 'normal')
    y_pos -= 0.45

# Bottom statement
ax.plot([1.5, 8.5], [1.8, 1.8], color='#1A237E', linewidth=1.5, zorder=3)
ax.text(5, 1.3, 'This assignment was determined by quantum mechanical measurement.', 
       ha='center', fontsize=9, style='italic', color='#555', zorder=3)
ax.text(5, 0.9, 'The CHSH violation (S > 2) proves no classical process produced these values.', 
       ha='center', fontsize=9, style='italic', color='#555', zorder=3)

plt.title('', fontsize=1)
plt.tight_layout()
plt.savefig('fairness_certificate.png', dpi=120, bbox_inches='tight')
plt.show()